# NVDA Robinhood Chain Phase 0 recovery

Bounded data-quality validation for the question: **Can Robinhood Chain Stock Token overnight trading contain information about the next US equity market open?**

This notebook reports feasibility only. It does not calculate equity returns, test prediction, define a strategy, or expand beyond NVDA.

## Fixed mapping and retrieval method

- Canonical NVDA Stock Token: `0xd0601CE157Db5bdC3162BbaC2a2C8aF5320D9EEC`
- NVDA/USDG Uniswap V3 pool: `0xd4eb21209c4d6093f80b5b84f5c45cc093ea14a3`
- USDG is token0 (6 decimals); NVDA is token1 (18 decimals).
- Executed price is reconstructed only from signed Uniswap V3 Swap amounts.
- Official Robinhood Chain RPC is the raw-log completeness source. Exact block timestamps use the bounded fallback RPC because the official endpoint throttled timestamp batches.
- Nineteen sessions use resumable adaptive chunk ledgers. The validated 2026-08-11 forensic session is reused without downloading it again.
- Current multiplier 1.0 is used only because no NVDA corporate action was effective in the sample window.

In [ ]:
import json
from pathlib import Path

import pandas as pd

project_root = Path.cwd() if (Path.cwd() / 'reports').exists() else Path.cwd().parent
result_dir = project_root / 'reports/robinhood_chain_phase0/nvda_20_session'
sessions = pd.read_csv(result_dir / 'nvda_session_quality.csv')
summary = json.loads((result_dir / 'summary.json').read_text(encoding='utf-8'))
summary

In [ ]:
# Compact session-level evidence; raw swaps remain in per-session checkpoint folders.
columns = [
    'market_open_date', 'swap_fill_count', 'active_trading_minutes',
    'usdg_notional', 'median_trade_size_usdg',
    'largest_execution_gap_minutes', 'reconstructable_price_percentage',
    'terminal_block_coverage_complete',
]
sessions[columns]

In [ ]:
# Share-ready integrity checks. The forensic reuse is explicitly identified.
assert len(sessions) == sessions['market_open_date'].nunique() == 20
assert sessions['completion_status'].eq('complete').all()
assert sessions['terminal_block_coverage_complete'].all()
assert not sessions['zero_trade_session'].any()
assert sessions['reused_forensic_artifact'].sum() == 1
assert sessions['swap_fill_count'].sum() == summary['total_swaps']
assert sessions['reconstructable_price_count'].sum() / sessions['swap_fill_count'].sum() == summary['price_reconstruction_success_rate']
'All aggregate integrity checks passed.'

In [ ]:
pd.Series({
    'sessions': f"{summary['sessions_completed']} / {summary['sessions_total']}",
    'nonempty_sessions': f"{summary['nonempty_sessions']} / {summary['sessions_total']}",
    'total_swaps': summary['total_swaps'],
    'median_swaps_per_session': summary['median_swaps_per_session'],
    'total_notional_usdg': summary['total_overnight_notional_usdg'],
    'median_session_notional_usdg': summary['median_session_notional_usdg'],
    'median_trade_size_usdg': summary['median_trade_size_usdg'],
    'total_active_minutes': summary['total_active_trading_minutes'],
    'median_active_minutes_per_session': summary['median_active_minutes_per_session'],
    'aggregate_active_minute_coverage': summary['aggregate_active_minute_coverage'],
    'median_session_active_minute_coverage': summary['median_session_active_minute_coverage'],
    'price_reconstruction_rate': summary['price_reconstruction_success_rate'],
    'sessions_with_gaps_ge_60m': summary['sessions_with_large_execution_gaps'],
    'incomplete_sessions': summary['retrieval_failures_or_incomplete_sessions'],
    'assessment': summary['assessment'],
})

## Phase 0 assessment: PASS

The bounded NVDA sample is sufficiently complete and active to proceed to a narrowly scoped overnight price-discovery pilot: all 20 sessions are nonempty, exact terminal ranges cover every session, executed prices reconstruct for about 99.99% of swaps, and no session has an execution gap of 60 minutes or more.

Limitations: this establishes data availability and activity, not predictive content. Economic trader counts remain unavailable because event sender/recipient fields may identify routers or hooks. USDG notional is pool-flow notional, not exchange-reported consolidated volume. The timestamp fallback was used only after official raw-log completeness was established. No additional token should be added without approval.